# ДЗ 5. CTE, recursive CTE, Materialized View: customer-level feature store на Olist

**Датасет:** Olist Brazilian E-commerce Public Dataset (Kaggle `olistbr/brazilian-ecommerce`), 5 таблиць: `customers`, `orders`, `order_items`, `products`, `sellers`. Повний обсяг.

План:
1. Setup: pgserver, отримання файлів, raw → typed, stub-fallback.
2. Feature pipeline як **CTE-chain з 9 named stages**: RFM, `cohort_quarter`, rolling AOV 30d, `cancellation_rate`, `preferred_category`.
3. Recursive CTE на синтетичному referral-графі `hw5_referrals`, із захистом від циклів і демонстрацією циклу A → B → A.
4. Materialized View з `UNIQUE INDEX` і `REFRESH ... CONCURRENTLY`, включно з демонстрацією staleness.
5. `EXPLAIN (ANALYZE, BUFFERS)`: повний pipeline проти lookup з MV, вартість кожної ознаки окремо, Reflection.

**Ключове рішення щодо грануляності.** Ознаки рахуються на рівні **реального покупця `customer_unique_id`**, а MV має по рядку на кожен `customer_id` з `UNIQUE INDEX` на ньому, як вимагає завдання. Причина: в Olist кожен `customer_id` має рівно одне замовлення (перевірено в 2.1). На рівні `customer_id` `frequency ≡ 1`, `cancellation_rate ∈ {0, 1}`, а rolling AOV просто дорівнює сумі замовлення, тобто ознаки вироджуються. Ключ `customer_id` в MV лишається зручним для serving: модель отримує замовлення з його `customer_id` і читає ознаки покупця одним index lookup.

## 1. Setup і typed-схема Olist

### 1.1. PostgreSQL через pgserver

In [1]:
# Install dependencies. PyPI ships pgserver wheels only for CPython 3.9-3.12, but its C extension
# is built against the stable ABI (abi3), so on Python 3.13+ we download the cp312 wheel and
# retag it as cp312-abi3, which pip accepts for any CPython >= 3.12.
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(PIP + ["psycopg2-binary", "sqlalchemy", "pandas", "kagglehub"], check=True)

if subprocess.run(PIP + ["pgserver"], capture_output=True).returncode != 0:
    wheel_dir = Path("/tmp/pgserver_wheel")
    subprocess.run(
        [sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
         "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
         "--platform", "manylinux2014_x86_64", "-d", str(wheel_dir)],
        check=True,
    )
    src = next(wheel_dir.glob("pgserver-0.1.4-cp312-cp312-*.whl"))
    dst = src.with_name(src.name.replace("cp312-cp312", "cp312-abi3"))
    src.rename(dst)
    subprocess.run(PIP + [str(dst)], check=True)

print(f"Python {sys.version.split()[0]}: залежності встановлено")

Python 3.13.16: залежності встановлено


In [2]:
import re
import statistics

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

pg = pgserver.get_server("/tmp/hw5_pg", cleanup_mode="stop")
# Pin psycopg2 explicitly: SQLAlchemy 2.1 maps "postgresql://" to psycopg 3, which is not installed
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)


def q(sql: str) -> pd.DataFrame:
    # Run a statement that returns rows and commit
    with engine.begin() as conn:
        return pd.read_sql(text(sql), conn)


def run_sql(sql: str) -> None:
    # Run a multi-statement SQL script in one transaction
    with engine.begin() as conn:
        conn.exec_driver_sql(sql)


def run_autocommit(sql: str) -> None:
    # Run a statement outside a transaction block (needed e.g. for VACUUM)
    with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        conn.exec_driver_sql(sql)


def explain(sql: str, show: bool = True, settings: str | None = None) -> tuple[str, float]:
    # Run EXPLAIN (ANALYZE, BUFFERS) and return the plan text and execution time in ms;
    # optional settings (e.g. "SET LOCAL work_mem = '64MB'") apply only inside this transaction
    with engine.begin() as conn:
        if settings:
            conn.exec_driver_sql(settings)
        rows = conn.exec_driver_sql(f"EXPLAIN (ANALYZE, BUFFERS) {sql}").fetchall()
    plan = "\n".join(r[0] for r in rows)
    exec_ms = float(re.search(r"Execution Time: ([\d.]+) ms", plan).group(1))
    if show:
        print(plan)
    return plan, exec_ms


with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### 1.2. Отримання файлів Olist

Джерела в порядку пріоритету: `data/olist/` → **KaggleHub** `olistbr/brazilian-ecommerce` → офіційний GitHub-репозиторій Olist [`olist/work-at-olist-data`](https://github.com/olist/work-at-olist-data/tree/master/datasets) (ті самі файли, без Kaggle credentials). Якщо жодне джерело недоступне, вмикається **stub-schema fallback** із завдання (`USE_STUB = True`). MD5 друкується для контролю ідентичності файлів.

In [3]:
import hashlib
from urllib.request import urlretrieve

FILES = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}
LOCAL_DIR = Path("data/olist")
GITHUB_BASE = "https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets"


def has_all_files(directory: Path) -> bool:
    # True if every required CSV is present in the directory
    return all((directory / name).exists() for name in FILES.values())


def resolve_dataset_dir() -> tuple[Path | None, str]:
    # Return the directory with the Olist CSVs (or None) and a label of the source used
    if has_all_files(LOCAL_DIR):
        return LOCAL_DIR, "локальна папка data/olist"
    try:
        import kagglehub

        kaggle_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
        if has_all_files(kaggle_dir):
            return kaggle_dir, "KaggleHub olistbr/brazilian-ecommerce"
    except Exception as exc:  # no credentials / no network
        print(f"KaggleHub недоступний ({type(exc).__name__}), пробую GitHub Olist")
    try:
        LOCAL_DIR.mkdir(parents=True, exist_ok=True)
        for name in FILES.values():
            urlretrieve(f"{GITHUB_BASE}/{name}", LOCAL_DIR / name)
        return LOCAL_DIR, "GitHub olist/work-at-olist-data"
    except Exception as exc:
        print(f"GitHub недоступний ({type(exc).__name__}): вмикаю stub-schema fallback")
        return None, "stub-schema"


dataset_dir, source_label = resolve_dataset_dir()
USE_STUB = dataset_dir is None
print(f"Джерело: {source_label}")
if not USE_STUB:
    for name in FILES.values():
        path = dataset_dir / name
        print(f"  {name:36s} {path.stat().st_size / 1e6:6.2f} MB  md5={hashlib.md5(path.read_bytes()).hexdigest()}")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Джерело: KaggleHub olistbr/brazilian-ecommerce
  olist_customers_dataset.csv            9.03 MB  md5=8a2c4244856aab4bde3b8ed81f8ca251
  olist_orders_dataset.csv              17.65 MB  md5=8bd60e55c1ca229d9f70b62f3e72f22c
  olist_order_items_dataset.csv         15.44 MB  md5=f4fa76976662e9cba0633f9e49ed2645
  olist_products_dataset.csv             2.38 MB  md5=e935401104a3669223b83c9a01fdf5ca
  olist_sellers_dataset.csv              0.17 MB  md5=5b22c02facdf842d6c0aea36e1849018


### 1.3. Raw-шар і typed-схема

Raw-таблиці читаються з `dtype=str`: це текстове дзеркало CSV, без вгадування типів pandas. Typed DDL взято із завдання без змін структури. Додано два індекси під access pattern pipeline: `orders(customer_id)` і `customers(customer_unique_id)`.

In [4]:
if not USE_STUB:
    for table_name, file_name in FILES.items():
        df = pd.read_csv(dataset_dir / file_name, dtype=str)
        df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000, method="multi")
        print(f"{table_name:24s} {df.shape}")

olist_customers_raw      (99441, 5)
olist_orders_raw         (99441, 8)
olist_order_items_raw    (112650, 7)
olist_products_raw       (32951, 9)
olist_sellers_raw        (3095, 4)


In [5]:
TYPED_DDL = r"""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals_cycle_demo CASCADE;
DROP TABLE IF EXISTS hw5_referrals     CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

-- Access-pattern indexes for the feature pipeline (customer -> orders, buyer identity)
CREATE INDEX idx_olist_orders_customer_id     ON olist_orders (customer_id);
CREATE INDEX idx_olist_customers_unique_id    ON olist_customers (customer_unique_id);
"""

LOAD_SQL = r"""
INSERT INTO olist_customers
SELECT customer_id, customer_unique_id, customer_zip_code_prefix::INTEGER, customer_city, customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id, customer_id, order_status,
    order_purchase_timestamp::TIMESTAMP, order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP, order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id, product_category_name,
    product_name_lenght::INTEGER, product_description_lenght::INTEGER,
    product_photos_qty::INTEGER, product_weight_g::INTEGER,
    product_length_cm::INTEGER, product_height_cm::INTEGER, product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT seller_id, seller_zip_code_prefix::INTEGER, seller_city, seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id, order_item_id::INTEGER, product_id, seller_id,
    shipping_limit_date::TIMESTAMP, price::NUMERIC(12, 2), freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
"""

STUB_SEED = r"""
INSERT INTO olist_sellers (seller_id, seller_zip_code_prefix, seller_city, seller_state)
VALUES
    ('s_01', 10001, 'Sao Paulo', 'SP'),
    ('s_02', 20001, 'Rio de Janeiro', 'RJ'),
    ('s_03', 30001, 'Belo Horizonte', 'MG');

INSERT INTO olist_products (
    product_id, product_category_name,
    product_name_length, product_description_length,
    product_photos_qty, product_weight_g,
    product_length_cm, product_height_cm, product_width_cm
)
VALUES
    ('p_01', 'electronics', 10, 100, 2,  800, 20, 10, 15),
    ('p_02', 'electronics', 12, 120, 3, 1200, 25, 12, 18),
    ('p_03', 'home',        8,  90, 1, 1500, 30, 20, 25),
    ('p_04', 'home',        9,  80, 2,  700, 18, 10, 14),
    ('p_05', 'fashion',     7,  60, 1,  300, 10,  5,  8),
    ('p_06', 'beauty',      6,  50, 1,  200,  8,  4,  6),
    ('p_07', 'sport',       9,  70, 2,  900, 20, 15, 16);

INSERT INTO olist_customers (
    customer_id, customer_unique_id, customer_zip_code_prefix, customer_state, customer_city
)
SELECT
    'c_' || LPAD(n::TEXT, 3, '0'),
    'u_' || LPAD(((n - 1) % 25 + 1)::TEXT, 3, '0'),
    10000 + n,
    (ARRAY['SP', 'RJ', 'MG', 'BA', 'PR'])[(n % 5) + 1],
    'City' || (n % 10)
FROM generate_series(1, 30) AS n;

INSERT INTO olist_orders (
    order_id, customer_id, order_status,
    order_purchase_timestamp, order_approved_at,
    order_delivered_carrier_date, order_delivered_customer_date,
    order_estimated_delivery_date
)
SELECT
    'o_' || LPAD(n::TEXT, 3, '0'),
    'c_' || LPAD(((n % 30) + 1)::TEXT, 3, '0'),
    (ARRAY['delivered', 'delivered', 'delivered', 'canceled', 'shipped'])[(n % 5) + 1],
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days'),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '1 day',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '3 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '8 days',
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '10 days'
FROM generate_series(1, 50) AS n;

INSERT INTO olist_order_items (
    order_id, order_item_id, product_id, seller_id,
    shipping_limit_date, price, freight_value
)
SELECT
    'o_' || LPAD(((n - 1) % 50 + 1)::TEXT, 3, '0'),
    ((n - 1) / 50) + 1,
    'p_0' || ((n % 7) + 1),
    's_0' || ((n % 3) + 1),
    TIMESTAMP '2024-01-01 00:00:00' + (n * INTERVAL '7 days') + INTERVAL '5 days',
    (n % 200 + 10)::NUMERIC(12, 2),
    (n % 30 + 5)::NUMERIC(12, 2)
FROM generate_series(1, 100) AS n;
"""

run_sql(TYPED_DDL)
run_sql(STUB_SEED if USE_STUB else LOAD_SQL)
run_autocommit("ANALYZE")  # fresh planner statistics for EXPLAIN
print("typed-схема:", "stub" if USE_STUB else "Olist")

typed-схема: Olist


### 1.4. Smoke-test кількості рядків

In [6]:
counts = q("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name
""")
if not USE_STUB:
    raw_counts = q("""
    SELECT 'customers' AS table_name, COUNT(*) AS raw_rows FROM olist_customers_raw
    UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders_raw
    UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items_raw
    UNION ALL SELECT 'products', COUNT(*) FROM olist_products_raw
    UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers_raw
    """)
    counts = counts.merge(raw_counts, on="table_name")
    assert (counts.n_rows == counts.raw_rows).all(), "typed-таблиця втратила рядки"
counts

,table_name,n_rows,raw_rows
0,customers,99441,99441
1,order_items,112650,112650
2,orders,99441,99441
3,products,32951,32951
4,sellers,3095,3095


**Висновок.** Усі п'ять typed-таблиць мають рівно стільки рядків, скільки raw: 99 441 клієнт і замовлення, 112 650 позицій, 32 951 товар, 3 095 продавців. Втрат немає: усі FK знайшли батьківські рядки, жоден `CHECK` не спрацював (інакше `INSERT` упав би помилкою).

## 2. Feature pipeline: CTE-chain з 9 named stages

### 2.1. Чому грануляність покупця, а не `customer_id`

In [7]:
grain = q("""
SELECT
    'customer_id' AS grain,
    COUNT(*) AS n_entities,
    MAX(n_orders) AS max_orders,
    COUNT(*) FILTER (WHERE n_orders >= 2) AS entities_with_2plus_orders
FROM (SELECT customer_id, COUNT(*) AS n_orders FROM olist_orders GROUP BY customer_id) AS a
UNION ALL
SELECT
    'customer_unique_id',
    COUNT(*),
    MAX(n_orders),
    COUNT(*) FILTER (WHERE n_orders >= 2)
FROM (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
) AS b
""")
grain

,grain,n_entities,max_orders,entities_with_2plus_orders
0,customer_id,99441,1,0
1,customer_unique_id,96096,17,2997


**Висновок.** На рівні `customer_id` максимум **1** замовлення і **0** сутностей з двома чи більше замовленнями. Будь-яка ознака частоти або повторної поведінки тут константа. На рівні `customer_unique_id` 96 096 покупців, **2 997** з них повторні, максимум **17** замовлень. Тому pipeline агрегує за покупцем.

### 2.2. Структура pipeline

| Stage | CTE | Грануляність виходу | Відповідальність |
|---|---|---|---|
| 0 | `reference_date` | 1 рядок | стабільна дата «станом на»: `MAX(purchase)::DATE + 1` |
| 1 | `stage1_orders_total` | замовлення | сума замовлення + статус + покупець (`customer_unique_id`) |
| 2 | `stage2_rfm` | покупець | recency / frequency / monetary, перша покупка |
| 3 | `stage3_cohort` | покупець | `cohort_quarter` з першої покупки |
| 4 | `stage4_rolling_aov` | замовлення | rolling AVG суми за 30 днів (window, без втрати рядків) + ранг найновішого |
| 5 | `stage5_cancel_rate` | покупець | частка скасованих замовлень |
| 6 | `stage6_category_revenue` | покупець × категорія | revenue за категорією |
| 7 | `stage7_preferred_category` | покупець | top-1 категорія (`ROW_NUMBER`) |
| 8 | `stage8_customer_features` | покупець | збирання всіх ознак в один рядок на покупця |
| — | фінальний `SELECT` | `customer_id` | ознаки покупця для кожного `customer_id` (ключ MV) |

Відмінності від шаблону, всі пов'язані лише з грануляністю: `PARTITION BY` / `GROUP BY` за `customer_unique_id` замість `customer_id` і окремий Stage 8. Формули ознак не змінено.

Pipeline зберігається в одній Python-змінній `FEATURE_PIPELINE` і без змін використовується в preview, у визначенні MV та в `EXPLAIN`. Так гарантовано, що MV — це той самий повний pipeline, а не його скорочена копія.

In [8]:
FEATURE_PIPELINE = r"""
WITH
reference_date AS (
    -- Stage 0: stable reference date for a historical dataset (not CURRENT_DATE)
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: order grain: order total, status and the real buyer (customer_unique_id)
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        c.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM per buyer; recency is measured from the stable reference date
    SELECT
        s.customer_unique_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_unique_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: acquisition cohort = quarter of the first order
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day AVG order value per buyer (window keeps order-level rows)
    SELECT
        s.customer_unique_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_unique_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: share of canceled orders per buyer
    SELECT
        customer_unique_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_unique_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue per buyer x product category (item grain -> buyer x category)
    SELECT
        c.customer_unique_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY c.customer_unique_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: top-1 category by revenue per buyer; ties broken by name for determinism
    SELECT
        customer_unique_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_unique_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
),
stage8_customer_features AS (
    -- Stage 8: assemble one feature row per buyer from the independent stages
    SELECT
        coh.customer_unique_id,
        coh.recency_days,
        coh.frequency,
        coh.monetary,
        coh.first_order_date,
        coh.cohort_quarter,
        aov.avg_order_value_30d,
        cancel.cancellation_rate,
        pc.product_category_name AS preferred_category,
        pc.category_revenue AS preferred_category_revenue
    FROM stage3_cohort AS coh
    LEFT JOIN stage4_rolling_aov AS aov
        ON aov.customer_unique_id = coh.customer_unique_id
       AND aov.rn_recent = 1
    LEFT JOIN stage5_cancel_rate AS cancel
        ON cancel.customer_unique_id = coh.customer_unique_id
    LEFT JOIN stage7_preferred_category AS pc
        ON pc.customer_unique_id = coh.customer_unique_id
)
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(f.recency_days, 9999) AS recency_days,
    COALESCE(f.frequency, 0) AS frequency,
    COALESCE(f.monetary, 0)::NUMERIC(12, 2) AS monetary,
    f.first_order_date,
    f.cohort_quarter,
    COALESCE(f.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(f.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(f.preferred_category, '(none)') AS preferred_category,
    COALESCE(f.preferred_category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage8_customer_features AS f
    ON f.customer_unique_id = c.customer_unique_id
"""

preview = q(FEATURE_PIPELINE + "\nORDER BY monetary DESC NULLS LAST, customer_id\nLIMIT 20")
preview

,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,349509b216bd5ec11c5fae929fd13595,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
2,df55c14d1476a9a3467f131269c2477f,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
3,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
4,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
5,f48d464a0baaea338cb25f816991ab1f,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
6,3fd6777bbce08a352fddd04e4a7cc8f6,ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
7,05455dfa7cd02f13d132aa7a6a9729c6,4007669dec559734d6f53e029e360987,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
8,e0a2412720e9ea4f26c1ac985f6a7358,5d0a2980b292d049061542014e8960bf,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
9,24bbf5fd2f2e1b359ee7de94defc4a15,eebb5dda148d3893cdaf5b5ca3040ccb,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34


**Висновок.**
- **Найбільший покупець** має одне замовлення на 13 664. Його `avg_order_value_30d_recent` = monetary, бо у 30-денному вікні лише одне замовлення.
- **Два рядки на одного покупця.** Покупець `da122df9…` трапляється двічі, бо має два `customer_id`. Обидва рядки містять ідентичні ознаки покупця (`frequency = 2`, rolling AOV 3 785,82 = середнє двох замовлень в один день). Це очікувана поведінка: MV має ключ `customer_id`, а ознаки належать покупцю.
- **Скасовані замовлення в `monetary`.** У рядку 8 `monetary` = 4 809,44 при `cancellation_rate = 1.0`: формула з шаблону включає вартість скасованих замовлень. Для моделі LTV чи churn це варто розділити на `monetary_delivered` і `monetary_canceled`.

### 2.3. Перевірка кожного stage окремо

Перевага CTE-chain у тому, що кожен stage можна перевірити ізольовано. Нижче — інваріанти, які мають виконуватися, якщо логіка stage коректна. Кожен рахується **незалежно від pipeline** з базових таблиць.

In [9]:
PIPE_CTES = FEATURE_PIPELINE.split("\nSELECT\n    c.customer_id,")[0]  # the WITH ... chain without the final SELECT

checks = q(PIPE_CTES + """
SELECT
    -- Stage 1: one row per order, totals equal the raw item sum
    (SELECT COUNT(*) FROM stage1_orders_total) = (SELECT COUNT(*) FROM olist_orders)                  AS s1_one_row_per_order,
    (SELECT SUM(order_total) FROM stage1_orders_total)
        = (SELECT SUM(price + freight_value) FROM olist_order_items)                                   AS s1_totals_match_items,
    -- Stage 2: frequency sums back to the number of orders; monetary sums back to total revenue
    (SELECT SUM(frequency) FROM stage2_rfm) = (SELECT COUNT(*) FROM olist_orders)                      AS s2_frequency_sums_to_orders,
    (SELECT SUM(monetary) FROM stage2_rfm)
        = (SELECT SUM(price + freight_value) FROM olist_order_items)                                   AS s2_monetary_sums_to_revenue,
    (SELECT MIN(recency_days) FROM stage2_rfm) >= 1                                                     AS s2_recency_positive,
    -- Stage 3: cohort is the first day of a quarter and not later than the first order
    (SELECT BOOL_AND(cohort_quarter <= first_order_date
                     AND EXTRACT(DAY FROM cohort_quarter) = 1
                     AND EXTRACT(MONTH FROM cohort_quarter) IN (1, 4, 7, 10)) FROM stage3_cohort)       AS s3_cohort_valid,
    -- Stage 4: window keeps all order rows; exactly one most recent order per buyer
    (SELECT COUNT(*) FROM stage4_rolling_aov) = (SELECT COUNT(*) FROM olist_orders)                    AS s4_no_rows_lost,
    (SELECT COUNT(*) FROM stage4_rolling_aov WHERE rn_recent = 1) = (SELECT COUNT(*) FROM stage2_rfm)  AS s4_one_recent_per_buyer,
    -- Stage 5: canceled share times frequency gives back the number of canceled orders
    (SELECT ROUND(SUM(cr.cancellation_rate * r.frequency)) FROM stage5_cancel_rate AS cr
        JOIN stage2_rfm AS r USING (customer_unique_id))
        = (SELECT COUNT(*) FROM olist_orders WHERE order_status = 'canceled')                          AS s5_cancel_count_matches,
    -- Stage 7: exactly one preferred category per buyer with items; it never exceeds monetary
    (SELECT COUNT(*) = COUNT(DISTINCT customer_unique_id) FROM stage7_preferred_category)              AS s7_one_category_per_buyer,
    (SELECT BOOL_AND(pc.category_revenue <= r.monetary) FROM stage7_preferred_category AS pc
        JOIN stage2_rfm AS r USING (customer_unique_id))                                               AS s7_le_monetary,
    -- Stage 8: one feature row per buyer
    (SELECT COUNT(*) FROM stage8_customer_features)
        = (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers)                              AS s8_one_row_per_buyer
""").T.rename(columns={0: "passed"})
assert checks.passed.all(), checks[~checks.passed]
checks

,passed
s1_one_row_per_order,True
s1_totals_match_items,True
s2_frequency_sums_to_orders,True
s2_monetary_sums_to_revenue,True
s2_recency_positive,True
s3_cohort_valid,True
s4_no_rows_lost,True
s4_one_recent_per_buyer,True
s5_cancel_count_matches,True
s7_one_category_per_buyer,True


Як працює rolling AOV 30d на реальному покупці з кількома замовленнями: `RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW` усереднює лише замовлення, що потрапили в 30-денне вікно перед поточним, і **не згортає** рядки.

In [10]:
example_buyer = q("""
SELECT c.customer_unique_id
FROM olist_orders AS o
JOIN olist_customers AS c ON c.customer_id = o.customer_id
GROUP BY c.customer_unique_id
HAVING COUNT(*) >= 4
   AND MAX(o.order_purchase_timestamp) - MIN(o.order_purchase_timestamp) > INTERVAL '60 days'
ORDER BY COUNT(*) DESC, c.customer_unique_id
LIMIT 1
""").customer_unique_id[0]

q(PIPE_CTES + f"""
SELECT order_ts, order_id, order_total, avg_order_value_30d, rn_recent
FROM stage4_rolling_aov
WHERE customer_unique_id = '{example_buyer}'
ORDER BY order_ts, order_id
""")

,order_ts,order_id,order_total,avg_order_value_30d,rn_recent
0,2017-05-15 23:30:03,5d848f3d93a493c1c8955e018240e7ca,22.77,22.77,17
1,2017-06-18 22:56:48,369634708db140c5d2c4e365882c443a,51.75,51.75,16
2,2017-07-17 22:11:13,5837a2c844decae8a778657425f6d664,0.00,25.88,15
3,2017-07-18 23:10:58,4f62d593acae92cea3c5662c76122478,21.77,10.89,14
4,2017-07-24 22:11:50,bf92c69b7cc70f7fc2c37de43e366173,166.71,62.83,13
5,2017-08-05 08:59:43,519203404f6116d406a970763ee75799,81.89,67.59,12
6,2017-09-05 22:14:52,e3071b7624445af6e4f3a1b23718667d,51.75,51.75,11
7,2017-10-18 23:25:04,cd4b336a02aacabd0ef22f6db711f95e,37.77,37.77,10
8,2017-10-29 16:58:02,89d9b111d2b990deb5f5f9769f92800b,21.77,29.77,9
9,2017-11-22 20:01:53,b850a16d8faf65a74c51287ef34379ce,21.77,21.77,8


**Висновок.** Покупець із 17 замовленнями показує, як працює вікно `RANGE ... 30 days PRECEDING`:
- **Самотнє замовлення** (2017-06-18): AOV = його сума 51,75.
- **Три замовлення за тиждень** (2017-07-17…24): AOV усереднює їх і дає 62,83.
- **Перерва понад 30 днів** (2017-09-05): вікно «скидається» до однієї суми.

Кількість рядків не змінюється (17 на вході, 17 на виході), а `rn_recent = 1` позначає останнє замовлення, з якого береться фінальна ознака.

**Data-quality:** замовлення від 2017-07-17 має `order_total = 0.00`. Це замовлення без позицій (скасоване чи недоступне), і воно занижує rolling AOV (25,88 замість 51,75). У production такі замовлення варто виключати з AOV.

### 2.4. Розподіли ознак

In [11]:
q(f"""
WITH f AS ({FEATURE_PIPELINE}),
buyers AS (SELECT DISTINCT ON (customer_unique_id) * FROM f ORDER BY customer_unique_id)
SELECT
    COUNT(*) AS n_buyers,
    COUNT(*) FILTER (WHERE frequency >= 2) AS repeat_buyers,
    COUNT(*) FILTER (WHERE cancellation_rate > 0) AS buyers_with_cancel,
    COUNT(*) FILTER (WHERE cancellation_rate > 0 AND cancellation_rate < 1) AS buyers_partial_cancel,
    COUNT(*) FILTER (WHERE recency_days = 9999) AS buyers_without_orders,
    COUNT(*) FILTER (WHERE preferred_category IN ('(none)', '(unknown)')) AS buyers_unknown_category,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY recency_days) AS median_recency_days,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY monetary) AS median_monetary
FROM buyers
""")

,n_buyers,repeat_buyers,buyers_with_cancel,buyers_partial_cancel,buyers_without_orders,buyers_unknown_category,median_recency_days,median_monetary
0,96096,2997,616,80,0,2041,270.0,107.27


In [12]:
q(f"""
WITH f AS ({FEATURE_PIPELINE}),
buyers AS (SELECT DISTINCT ON (customer_unique_id) * FROM f ORDER BY customer_unique_id)
SELECT
    cohort_quarter,
    COUNT(*) AS n_buyers,
    ROUND(AVG(frequency), 3) AS avg_frequency,
    ROUND(AVG(monetary), 2) AS avg_monetary,
    ROUND(AVG(cancellation_rate) * 100, 2) AS pct_cancel
FROM buyers
GROUP BY cohort_quarter
ORDER BY cohort_quarter
""")

,cohort_quarter,n_buyers,avg_frequency,avg_monetary,pct_cancel
0,2016-07-01,4,1.000,88.69,50.00
1,2016-10-01,322,1.043,181.98,7.45
2,2017-01-01,5152,1.059,171.08,1.03
3,2017-04-01,9087,1.061,170.45,0.63
4,2017-07-01,12208,1.055,164.08,0.58
5,2017-10-01,17261,1.036,163.26,0.42
6,2018-01-01,20441,1.033,158.51,0.61
7,2018-04-01,19273,1.018,169.92,0.29
8,2018-07-01,12347,1.013,163.47,0.93
9,2018-10-01,1,1.000,0.00,100.00


**Висновок.**
- **Повторні покупці:** 2 997 з 96 096 (3,1 %).
- **Скасування:** хоча б одне скасоване замовлення мають 616 покупців. Лише у **80** з них `0 < cancellation_rate < 1`, у решти скасоване все. Тож на цьому датасеті ознака майже бінарна.
- **`COALESCE(..., 9999)` не спрацював жодного разу:** в Olist кожен клієнт має замовлення.
- **2 051** покупців мають preferred category `(unknown)` (товар без категорії) або `(none)` (замовлення без позицій).
- **Когорти:** найбільші — 2018-Q1 і Q2. Частка скасувань найвища в найранішніх когортах (2016-Q4: 7,45 %).
- **Артефакт `reference_date`.** Когорта 2018-10 — це **1** покупець із `monetary = 0` і 100 % скасувань. Саме його скасоване замовлення без позицій (2018-10-17) визначає `MAX(order_purchase_timestamp)`. Тому `reference_date` зсунуто на ~1,5 місяця після реального кінця активності, і медіанна recency (270 днів) включає цей «мертвий» період. Стабільнішою була б дата з доставлених замовлень або фіксований business cutoff.

## 3. Recursive CTE: referral chain

### 3.1. Синтетичний referral-граф `hw5_referrals`

Olist не має referral-даних, тому `hw5_referrals` — **навчальна імітація**. Замість ланцюжка з 10 вузлів (як у шаблоні) будуємо детерміноване **бінарне дерево** з 31 клієнта (перші за `customer_id`). Клієнт з номером `rn` запрошений клієнтом `rn / 2`, тож дерево має 5 рівнів (1 + 2 + 4 + 8 + 16). На ланцюжку кожен рівень має рівно 1 вузол; дерево показує, що recursive CTE коректно розгалужується.

`referred_at` задано детерміновано (а не `DEFAULT NOW()`), щоб результат не залежав від моменту запуску. `PRIMARY KEY (referred_customer_id)` гарантує, що кожного клієнта запросили не більше одного разу (in-degree ≤ 1), тобто це ліс дерев. `ON CONFLICT DO NOTHING` робить seed ідемпотентним.

In [13]:
run_sql("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

CREATE INDEX idx_hw5_referrals_referrer ON hw5_referrals (referrer_customer_id);
""")

REFERRAL_SEED = """
WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 31
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id, referred_at)
SELECT
    parent.customer_id AS referrer_customer_id,
    child.customer_id  AS referred_customer_id,
    TIMESTAMPTZ '2018-01-01 00:00:00+00' + child.rn * INTERVAL '1 day' AS referred_at
FROM ordered_customers AS child
JOIN ordered_customers AS parent
    ON parent.rn = child.rn / 2
WHERE child.rn >= 2
ON CONFLICT (referred_customer_id) DO NOTHING
RETURNING referred_customer_id
"""
seed_1 = q(REFERRAL_SEED)
seed_2 = q(REFERRAL_SEED)  # idempotency check
print(f"Перший seed вставив: {len(seed_1)} ребер, повторний: {len(seed_2)}")
q("SELECT * FROM hw5_referrals ORDER BY referred_at LIMIT 6")

Перший seed вставив: 30 ребер, повторний: 0


,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2018-01-03 00:00:00+00:00
1,00012a2ce6f8dcda20d059ce98491703,0001fd6190edaaf884bcaf3d49edf079,2018-01-04 00:00:00+00:00
2,000161a058600d5901f007fab4c27140,0002414f95344307404f0ace7a26f1d5,2018-01-05 00:00:00+00:00
3,000161a058600d5901f007fab4c27140,000379cdec625522490c315e70c7a9fb,2018-01-06 00:00:00+00:00
4,0001fd6190edaaf884bcaf3d49edf079,0004164d20a9e969af783496f3408652,2018-01-07 00:00:00+00:00
5,0001fd6190edaaf884bcaf3d49edf079,000419c5494106c306a97b5635748086,2018-01-08 00:00:00+00:00


### 3.2. Recursive CTE: depth і path

- **Anchor:** корені дерева — клієнти, які **запрошували** когось (`EXISTS`), але самі **не були запрошені** (`NOT EXISTS`). Умову `EXISTS` додано до шаблону: без неї всі ~99 тис. клієнтів поза програмою стали б «коренями» глибини 1 і заглушили б результат.
- **Recursive part:** перехід від referrer до його referred customers.
- **Termination:** рекурсія закінчується, коли recursive part повертає 0 рядків (у листків немає дітей). Плюс два явні захисти: `NOT r.referred_customer_id = ANY(rd.path_ids)` (вузол уже в шляху, тобто цикл) і `rd.depth < 50` (depth guard).

Чому `NOT EXISTS`, а не `NOT IN`: якщо підзапит `NOT IN (SELECT referred_customer_id ...)` містить хоч один `NULL`, вираз дає `NULL` для **кожного** рядка і anchor стає порожнім. `NOT EXISTS` — справжній anti-join, нечутливий до `NULL`.

In [14]:
RECURSIVE_SQL = r"""
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: roots = customers who referred someone but were not referred themselves
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referrer_customer_id = c.customer_id
    )
      AND NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: move from a referrer to the customers they referred
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)   -- cycle guard
      AND rd.depth < 50                                    -- depth guard
)
"""

by_depth = q(RECURSIVE_SQL + """
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(LEFT(path_text, 8) || '…' || RIGHT(path_text, 8)) AS sample_path_short,
    MAX(CARDINALITY(path_ids)) AS path_len
FROM referral_depth
GROUP BY depth
ORDER BY depth
""")
by_depth

,depth,n_customers,sample_path_short,path_len
0,1,1,00012a2c…98491703,1
1,2,2,00012a2c…b4c27140,2
2,3,4,00012a2c…f3408652,3
3,4,8,00012a2c…ff1a2be5,4
4,5,16,00012a2c…daf1b0fe,5


In [15]:
# Full path for one deepest node, with short ids for readability
q(RECURSIVE_SQL + """
SELECT
    depth,
    ARRAY_TO_STRING(ARRAY(SELECT LEFT(x, 6) FROM UNNEST(path_ids) AS x), ' -> ') AS path_short_ids
FROM referral_depth
WHERE depth = (SELECT MAX(depth) FROM referral_depth)
ORDER BY path_text
LIMIT 3
""")

,depth,path_short_ids
0,5,00012a -> 000161 -> 000241 -> 00046a -> 000bf8
1,5,00012a -> 000161 -> 000241 -> 00046a -> 000e94
2,5,00012a -> 000161 -> 000241 -> 00050b -> 000f17


In [16]:
# Coverage check: every node of the graph is reached exactly once
q(RECURSIVE_SQL + """
SELECT
    (SELECT COUNT(*) FROM referral_depth)                                AS reached_rows,
    (SELECT COUNT(DISTINCT customer_id) FROM referral_depth)             AS reached_distinct,
    (SELECT COUNT(*) FROM (
        SELECT referrer_customer_id FROM hw5_referrals
        UNION
        SELECT referred_customer_id FROM hw5_referrals) AS nodes)        AS nodes_in_graph,
    (SELECT COUNT(*) FROM hw5_referrals AS r
     WHERE NOT EXISTS (SELECT 1 FROM referral_depth AS d
                       WHERE d.customer_id = r.referred_customer_id))   AS unreachable_referred
""")

,reached_rows,reached_distinct,nodes_in_graph,unreachable_referred
0,31,31,31,0


**Висновок.**
- **Дерево:** 31 вузол, 5 рівнів по 1 / 2 / 4 / 8 / 16 клієнтів, як і задумано (бінарне дерево). Довжина `path_ids` дорівнює `depth`.
- **Ідемпотентність seed:** перший запуск вставив 30 ребер, повторний — 0 (`ON CONFLICT DO NOTHING`).
- **Покриття:** досягнуто 31 рядок і 31 унікальний вузол, `unreachable_referred = 0`. Кожен вузол знайдено рівно один раз: дублікатів шляхів немає, бо in-degree ≤ 1 (PK на `referred_customer_id`).
- **Termination:** рекурсія завершилась природно на глибині 5, коли у листків не виявилося дітей. Жоден guard не знадобився, але вони страхують від некоректних даних.

### 3.3. Що відбувається з циклом A → B → A

Цикл демонструємо в **окремій** таблиці `hw5_referrals_cycle_demo`, щоб не псувати основний граф: A → B, B → A і ще B → C. Три сценарії:

1. **Anchor як у 3.2** (корені без вхідних ребер). У циклі A і B обидва «запрошені», тож коренів **немає**, і CTE повертає 0 рядків. Рекурсія завершується, але цикл **мовчки зникає** з результату. Тому в production потрібна окрема перевірка «вузли, не досяжні з жодного кореня».
2. **Anchor = конкретний клієнт A** (наприклад, «покажи ланцюжок від A»). Тут цикл досяжний: з path-guard рекурсія зупиняється, щойно наступний вузол уже є в `path_ids`.
3. **Той самий запит без path-guard, лише з depth guard `< 6`.** Рекурсія теж завершується, але генерує «сміттєві» рядки A, B, A, B… аж до ліміту глибини. Depth guard лише обмежує шкоду, а не виявляє цикл.

Наостанок — вбудований `CYCLE`-clause PostgreSQL 14+: він сам веде шлях і позначає рядок, що закриває цикл.

In [17]:
run_sql("""
DROP TABLE IF EXISTS hw5_referrals_cycle_demo CASCADE;
CREATE TABLE hw5_referrals_cycle_demo (LIKE hw5_referrals INCLUDING ALL);
""")

abc = q("SELECT customer_id FROM olist_customers ORDER BY customer_id DESC LIMIT 3").customer_id.tolist()
A, B, C = abc
run_sql(f"""
INSERT INTO hw5_referrals_cycle_demo (referrer_customer_id, referred_customer_id, referred_at) VALUES
    ('{A}', '{B}', TIMESTAMPTZ '2018-02-01 00:00:00+00'),
    ('{B}', '{A}', TIMESTAMPTZ '2018-02-02 00:00:00+00'),
    ('{B}', '{C}', TIMESTAMPTZ '2018-02-03 00:00:00+00');
""")
short = {A: "A", B: "B", C: "C"}
print({v: k[:10] + "…" for k, v in short.items()})


def cycle_query(anchor: str, path_guard: bool, depth_limit: int) -> str:
    # Recursive CTE over the cycle demo table with configurable guards
    guard = "AND NOT r.referred_customer_id = ANY(rd.path_ids)" if path_guard else ""
    return f"""
    WITH RECURSIVE chain AS (
        SELECT c.customer_id, 1 AS depth, ARRAY[c.customer_id]::TEXT[] AS path_ids
        FROM olist_customers AS c
        WHERE {anchor}
        UNION ALL
        SELECT r.referred_customer_id, rd.depth + 1, rd.path_ids || r.referred_customer_id
        FROM hw5_referrals_cycle_demo AS r
        JOIN chain AS rd ON rd.customer_id = r.referrer_customer_id
        WHERE rd.depth < {depth_limit} {guard}
    )
    SELECT depth, customer_id, path_ids FROM chain ORDER BY depth, customer_id
    """


def readable(df: pd.DataFrame) -> pd.DataFrame:
    # Replace long customer ids with A/B/C labels
    out = df.copy()
    out["customer_id"] = out.customer_id.map(short)
    out["path"] = out.path_ids.map(lambda p: " -> ".join(short.get(x, "?") for x in p))
    return out.drop(columns="path_ids")


roots_anchor = """EXISTS (SELECT 1 FROM hw5_referrals_cycle_demo r WHERE r.referrer_customer_id = c.customer_id)
                  AND NOT EXISTS (SELECT 1 FROM hw5_referrals_cycle_demo r WHERE r.referred_customer_id = c.customer_id)"""
s1 = q(cycle_query(roots_anchor, path_guard=True, depth_limit=50))
print(f"(1) Anchor = корені без вхідних ребер: {len(s1)} рядків\n")

s2 = readable(q(cycle_query(f"c.customer_id = '{A}'", path_guard=True, depth_limit=50)))
print("(2) Anchor = A, path-guard + depth-guard:")
display(s2)

s3 = readable(q(cycle_query(f"c.customer_id = '{A}'", path_guard=False, depth_limit=6)))
print("(3) Anchor = A, БЕЗ path-guard, лише depth < 6:")
display(s3)

{'A': 'ffffe8b65b…', 'B': 'ffffa31725…', 'C': 'ffff42319e…'}
(1) Anchor = корені без вхідних ребер: 0 рядків

(2) Anchor = A, path-guard + depth-guard:


,depth,customer_id,path
0,1,A,A
1,2,B,A -> B
2,3,C,A -> B -> C


(3) Anchor = A, БЕЗ path-guard, лише depth < 6:


,depth,customer_id,path
0,1,A,A
1,2,B,A -> B
2,3,C,A -> B -> C
3,3,A,A -> B -> A
4,4,B,A -> B -> A -> B
5,5,C,A -> B -> A -> B -> C
6,5,A,A -> B -> A -> B -> A
7,6,B,A -> B -> A -> B -> A -> B


In [18]:
# Built-in CYCLE clause (PostgreSQL 14+): tracks the path itself and flags the closing row
cycle_clause = q(f"""
WITH RECURSIVE chain AS (
    SELECT c.customer_id, 1 AS depth
    FROM olist_customers AS c
    WHERE c.customer_id = '{A}'
    UNION ALL
    SELECT r.referred_customer_id, ch.depth + 1
    FROM hw5_referrals_cycle_demo AS r
    JOIN chain AS ch ON ch.customer_id = r.referrer_customer_id
) CYCLE customer_id SET is_cycle USING cycle_path
SELECT depth, customer_id, is_cycle, cycle_path::TEXT AS cycle_path
FROM chain
ORDER BY depth, customer_id
""")
cycle_clause["customer_id"] = cycle_clause.customer_id.map(short)
cycle_clause["cycle_path"] = cycle_clause.cycle_path.map(
    lambda p: " -> ".join(short.get(x, "?") for x in re.findall(r"\(([^)]+)\)", p))
)
cycle_clause

,depth,customer_id,is_cycle,cycle_path
0,1,A,False,A
1,2,B,False,A -> B
2,3,C,False,A -> B -> C
3,3,A,True,A -> B -> A


**Висновок за циклом A → B → A (+ B → C):**
1. **Anchor «корені без вхідних ребер» повертає 0 рядків.** У циклі кожен вузол має вхідне ребро, тож коренів немає. Рекурсія коректно завершується, але цикл **невидимий**. Оскільки PK `referred_customer_id` обмежує in-degree до 1, цикл у цій схемі ніколи не досяжний з кореня. Головний ризик тут не нескінченна рекурсія, а **тихо загублені вузли**. Їх ловить перевірка покриття з 3.2 (`unreachable_referred`).
2. **Anchor = A, з path-guard:** рядки A → B → C, а ребро B → A відкинуто умовою `NOT ... = ANY(path_ids)`. Рекурсія зупинилась на глибині 3 без жодного дубля.
3. **Без path-guard, лише `depth < 6`:** запит теж завершився, але повернув **8** рядків замість 3. A і B повторюються по колу (A → B → A → B …) до ліміту глибини. Depth guard гарантує термінацію, але не коректність.
4. **`CYCLE customer_id SET is_cycle USING cycle_path`** дає той самий результат, що й ручний path-guard, і додатково позначає рядок, що закриває цикл (`is_cycle = True` для A → B → A). Цикл виявлено явно, а не просто відфільтровано.

## 4. Materialized View з `REFRESH ... CONCURRENTLY`

MV створюється з **тієї самої** змінної `FEATURE_PIPELINE`, тобто з повного 9-stage pipeline. `UNIQUE INDEX` на `customer_id` — обов'язкова умова для `REFRESH ... CONCURRENTLY`: PostgreSQL за цим ключем порівнює стару й нову версії MV і застосовує лише різницю (`INSERT` / `UPDATE` / `DELETE`), не блокуючи читачів.

In [19]:
run_sql(f"""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
{FEATURE_PIPELINE};

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
""")
run_autocommit("ANALYZE mv_olist_customer_features")

q("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(DISTINCT customer_unique_id) AS distinct_buyers,
    (SELECT COUNT(*) FROM olist_customers) AS customers_in_source,
    PG_SIZE_PRETTY(PG_TOTAL_RELATION_SIZE('mv_olist_customer_features')) AS mv_size_with_indexes
FROM mv_olist_customer_features
""")

,n_rows,distinct_customer_id,distinct_buyers,customers_in_source,mv_size_with_indexes
0,99441,99441,96096,99441,22 MB


In [20]:
q("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5
""")

,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,349509b216bd5ec11c5fae929fd13595,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
2,df55c14d1476a9a3467f131269c2477f,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,2,7571.63,2017-04-01,2017-04-01,3785.82,0.0,eletroportateis,7571.63
3,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
4,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31


In [21]:
# The MV must hold exactly the pipeline result: compare with a fresh recomputation (both directions)
diff = q(f"""
WITH fresh AS ({FEATURE_PIPELINE})
SELECT
    (SELECT COUNT(*) FROM (SELECT * FROM fresh EXCEPT SELECT * FROM mv_olist_customer_features) AS a) AS only_in_pipeline,
    (SELECT COUNT(*) FROM (SELECT * FROM mv_olist_customer_features EXCEPT SELECT * FROM fresh) AS b) AS only_in_mv
""")
assert diff.only_in_pipeline[0] == 0 and diff.only_in_mv[0] == 0
diff

,only_in_pipeline,only_in_mv
0,0,0


### 4.1. `REFRESH MATERIALIZED VIEW CONCURRENTLY`

In [22]:
import time

t0 = time.perf_counter()
run_sql("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;")
concurrent_s = time.perf_counter() - t0
print(f"REFRESH ... CONCURRENTLY виконано без помилок за {concurrent_s:.2f} с")

t0 = time.perf_counter()
run_sql("REFRESH MATERIALIZED VIEW mv_olist_customer_features;")
plain_s = time.perf_counter() - t0
print(f"Звичайний REFRESH (з ексклюзивним блокуванням читачів): {plain_s:.2f} с")

REFRESH ... CONCURRENTLY виконано без помилок за 7.49 с
Звичайний REFRESH (з ексклюзивним блокуванням читачів): 3.97 с


### 4.2. Freshness: MV застаріває до наступного refresh

Демонстрація на одному покупці: змінюємо статус одного його замовлення на `canceled` у базовій таблиці. View перерахувалась би одразу, а MV показує старе значення, доки не виконано `REFRESH`. Наприкінці зміну скасовуємо й оновлюємо MV, щоб наступні клітинки працювали на вихідних даних.

In [23]:
demo_customer, demo_order = q("""
SELECT o.customer_id, o.order_id
FROM olist_orders AS o
WHERE o.order_status = 'delivered'
ORDER BY o.order_id
LIMIT 1
""").iloc[0]

FEATURE_COLS = "customer_id, cancellation_rate, frequency, recency_days"


def snapshot(label: str) -> pd.DataFrame:
    # Compare the cached MV row with a fresh recomputation for the demo customer
    mv = q(f"SELECT {FEATURE_COLS} FROM mv_olist_customer_features WHERE customer_id = '{demo_customer}'")
    fresh = q(f"WITH f AS ({FEATURE_PIPELINE}) SELECT {FEATURE_COLS} FROM f WHERE customer_id = '{demo_customer}'")
    mv.insert(0, "source", "MV")
    fresh.insert(0, "source", "pipeline (fresh)")
    out = pd.concat([mv, fresh], ignore_index=True)
    out.insert(0, "step", label)
    return out


steps = [snapshot("0: вихідний стан")]
run_sql(f"UPDATE olist_orders SET order_status = 'canceled' WHERE order_id = '{demo_order}';")
steps.append(snapshot("1: після UPDATE, до REFRESH"))
run_sql("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;")
steps.append(snapshot("2: після REFRESH CONCURRENTLY"))

# Revert the demo change and refresh again so the rest of the notebook uses original data
run_sql(f"UPDATE olist_orders SET order_status = 'delivered' WHERE order_id = '{demo_order}';")
run_sql("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;")
steps.append(snapshot("3: зміну скасовано + REFRESH"))
pd.concat(steps, ignore_index=True)

,step,source,customer_id,cancellation_rate,frequency,recency_days
0,0: вихідний стан,MV,3ce436f183e68e07877b285a838db11a,0.0,1,400
1,0: вихідний стан,pipeline (fresh),3ce436f183e68e07877b285a838db11a,0.0,1,400
2,"1: після UPDATE, до REFRESH",MV,3ce436f183e68e07877b285a838db11a,0.0,1,400
3,"1: після UPDATE, до REFRESH",pipeline (fresh),3ce436f183e68e07877b285a838db11a,1.0,1,400
4,2: після REFRESH CONCURRENTLY,MV,3ce436f183e68e07877b285a838db11a,1.0,1,400
5,2: після REFRESH CONCURRENTLY,pipeline (fresh),3ce436f183e68e07877b285a838db11a,1.0,1,400
6,3: зміну скасовано + REFRESH,MV,3ce436f183e68e07877b285a838db11a,0.0,1,400
7,3: зміну скасовано + REFRESH,pipeline (fresh),3ce436f183e68e07877b285a838db11a,0.0,1,400


**Висновок.**
- **Розмір і ключ:** MV містить **99 441** рядок (по одному на `customer_id`, 96 096 покупців) і займає ~22 MB разом з індексом. `EXCEPT` в обидва боки дав **0 / 0**, тобто MV ідентична свіжому перерахунку pipeline.
- **Вартість refresh:** `REFRESH ... CONCURRENTLY` виконано без помилок за ~3–3,6 с, звичайний `REFRESH` — за ~1,8–2,3 с (між прогонами час коливається). Concurrent повільніший, бо будує нову версію в тимчасовій таблиці й порівнює її зі старою за UNIQUE INDEX. Зате під час refresh читачі не блокуються. Звичайний `REFRESH` бере `ACCESS EXCLUSIVE` lock, тож усі lookup-и чекали б.
- **Staleness:** після `UPDATE` статусу свіжий pipeline одразу показує `cancellation_rate = 1.0`, а MV — старе **0.0**. Лише після refresh значення збігаються. Скасування зміни теж потрапило в MV тільки після наступного refresh.

**Ризик повного переписування.** `reference_date` залежить від `MAX(order_purchase_timestamp)`. Одне нове замовлення зсуне дату, і recency зміниться для **всіх** 99 тис. рядків, тож concurrent refresh перепише всю MV, а не лише змінених покупців.

## 5. `EXPLAIN (ANALYZE, BUFFERS)`

### 5.1. Повний pipeline (усі 9 stages, top-10 за monetary)

На відміну від шаблону, де під `EXPLAIN` стояли лише Stage 0–2, тут аналізується **повний** pipeline з усіма ознаками. Тільки так порівняння з MV чесне: MV зберігає саме повний результат.

In [24]:
FULL_SQL = FEATURE_PIPELINE + "\nORDER BY monetary DESC NULLS LAST\nLIMIT 10"
full_plan, full_ms = explain(FULL_SQL)

Limit  (cost=144333.98..144334.01 rows=10 width=183) (actual time=3688.509..3698.385 rows=10 loops=1)
  Buffers: shared hit=15351, temp read=10662 written=10820
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=18906.55..30652.01 rows=112650 width=103) (actual time=471.824..758.516 rows=99441 loops=1)
          Group Key: c_2.customer_unique_id, o_1.order_id
          Buffers: shared hit=5609, temp read=1546 written=1550
          ->  Gather Merge  (cost=18906.55..28017.99 rows=66265 width=115) (actual time=471.809..634.747 rows=99681 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=5609, temp read=1546 written=1550
                ->  Partial GroupAggregate  (cost=17906.54..19563.17 rows=66265 width=115) (actual time=464.948..589.598 rows=49840 loops=2)
                      Group Key: c_2.customer_unique_id, o_1.order_id
                      Buffers: shared hit=5609, temp read=1546 written=1550
       

### 5.2. Lookup з MV за `customer_id` (UNIQUE INDEX)

In [25]:
MV_LOOKUP_SQL = """
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
"""
lookup_plan, lookup_ms = explain(MV_LOOKUP_SQL)

Nested Loop  (cost=0.83..8.91 rows=1 width=125) (actual time=0.023..0.024 rows=1 loops=1)
  Buffers: shared hit=8
  ->  Limit  (cost=0.42..0.47 rows=1 width=33) (actual time=0.012..0.013 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..5328.03 rows=99441 width=33) (actual time=0.011..0.011 rows=1 loops=1)
              Heap Fetches: 0
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=125) (actual time=0.007..0.007 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=4
Planning:
  Buffers: shared hit=6 read=4
Planning Time: 0.421 ms
Execution Time: 0.048 ms


### 5.3. Top-N з MV: пастка `NULLS LAST` в індексі з шаблону

Індекс `(monetary DESC)` має неявний порядок **`DESC NULLS FIRST`**, а запит просить `ORDER BY monetary DESC NULLS LAST`. Порядки не збігаються, тож planner **не може** використати індекс. Колонки MV не мають `NOT NULL`, і planner не знає, що `NULL` тут неможливі. Порівнюємо три плани: без індексу, з індексом із шаблону, з виправленим індексом.

In [26]:
TOPN_SQL = """
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
"""
print("=== (а) лише UNIQUE INDEX на customer_id ===")
topn_plan_a, topn_ms_a = explain(TOPN_SQL)

run_sql("CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary ON mv_olist_customer_features (monetary DESC);")
run_autocommit("ANALYZE mv_olist_customer_features")
print("\n=== (б) + індекс із шаблону: (monetary DESC) ===")
topn_plan_b, topn_ms_b = explain(TOPN_SQL)

run_sql("""
DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;
CREATE INDEX idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC NULLS LAST);
""")
run_autocommit("ANALYZE mv_olist_customer_features")
print("\n=== (в) виправлений індекс: (monetary DESC NULLS LAST) ===")
topn_plan_c, topn_ms_c = explain(TOPN_SQL)

=== (а) лише UNIQUE INDEX на customer_id ===
Limit  (cost=4961.01..4962.16 rows=10 width=125) (actual time=29.791..31.453 rows=10 loops=1)
  Buffers: shared hit=2148
  ->  Gather Merge  (cost=4961.01..11687.94 rows=58495 width=125) (actual time=29.789..31.449 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=2148
        ->  Sort  (cost=3961.00..4107.24 rows=58495 width=125) (actual time=26.079..26.080 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 29kB
              Buffers: shared hit=2148
              Worker 0:  Sort Method: top-N heapsort  Memory: 28kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2696.95 rows=58495 width=125) (actual time=0.012..9.940 rows=49720 loops=2)
                    Buffers: shared hit=2112
Planning Time: 0.117 ms
Execution Time: 31.498 ms

=== (б) + індекс із шаблону: (monetary DESC) ===
Limit  (cost=4961

### 5.4. Яка ознака найдорожча: вартість кожного stage окремо

Плани повного pipeline складно розкласти на ознаки вручну, тому кожен feature-stage вимірюється окремим `EXPLAIN ANALYZE` (`SELECT * FROM stageN`, мінімум з 5 запусків, бо він стійкіший до шуму, ніж середнє). Stage 1 — спільна база для 2, 4 і 5, тому для них наведено ще й **приріст** над Stage 1. Stage 6–7 читають базові таблиці самостійно.

In [27]:
def stage_cost(stage: str, runs: int = 5) -> tuple[float, str]:
    # Minimum execution time over several runs (robust to timing noise) and the top plan nodes
    sql = PIPE_CTES + f"\nSELECT * FROM {stage}"
    times, plan = [], ""
    for _ in range(runs):
        plan, ms = explain(sql, show=False)
        times.append(ms)
    nodes = [line.strip().split("  (")[0].lstrip("-> ").strip()
             for line in plan.splitlines() if "actual time" in line]
    return min(times), " | ".join(dict.fromkeys(nodes[:6]))


STAGES = {
    "stage1_orders_total": "order_total (база)",
    "stage2_rfm": "RFM",
    "stage3_cohort": "cohort_quarter",
    "stage4_rolling_aov": "rolling AOV 30d",
    "stage5_cancel_rate": "cancellation_rate",
    "stage7_preferred_category": "preferred_category (stage 6 + 7)",
    "stage8_customer_features": "усі ознаки покупця (stage 8)",
}
rows = []
for stage, feature in STAGES.items():
    ms, nodes = stage_cost(stage)
    rows.append({"stage": stage, "feature": feature, "min_ms": round(ms, 1), "top_plan_nodes": nodes})
stage_costs = pd.DataFrame(rows)
base_ms = stage_costs.loc[stage_costs.stage == "stage1_orders_total", "min_ms"].iloc[0]
stage_costs["over_stage1_ms"] = stage_costs.apply(
    lambda r: round(r.min_ms - base_ms, 1)
    if r.stage in ("stage2_rfm", "stage3_cohort", "stage4_rolling_aov", "stage5_cancel_rate") else None,
    axis=1,
)
stage_costs[["stage", "feature", "min_ms", "over_stage1_ms", "top_plan_nodes"]]

,stage,feature,min_ms,over_stage1_ms,top_plan_nodes
0,stage1_orders_total,order_total (база),839.9,NaN,CTE Scan on stage1_orders_total | Finalize GroupAggregate | Gather Merge | P...
1,stage2_rfm,RFM,1087.0,247.1,CTE Scan on stage2_rfm | Finalize GroupAggregate | Gather Merge | Partial Gr...
2,stage3_cohort,cohort_quarter,1120.2,280.3,CTE Scan on stage3_cohort | Finalize GroupAggregate | Gather Merge | Partial...
3,stage4_rolling_aov,rolling AOV 30d,1358.3,518.4,CTE Scan on stage4_rolling_aov | Finalize GroupAggregate | Gather Merge | Pa...
4,stage5_cancel_rate,cancellation_rate,1038.2,198.3,CTE Scan on stage5_cancel_rate | Finalize GroupAggregate | Gather Merge | Pa...
5,stage7_preferred_category,preferred_category (stage 6 + 7),919.8,NaN,CTE Scan on stage7_preferred_category | Subquery Scan on ranked | WindowAgg ...
6,stage8_customer_features,усі ознаки покупця (stage 8),3225.4,NaN,Hash Left Join | Finalize GroupAggregate | Gather Merge | Partial GroupAggre...


In [28]:
def plan_lines(plan: str, *keywords: str) -> list[str]:
    # Lines of the plan that contain any of the keywords (sorts, spills, scans)
    return [l.strip() for l in plan.splitlines() if any(k in l for k in keywords)]


summary = pd.DataFrame([
    {"scenario": "full pipeline (9 stages)", "execution_ms": full_ms},
    {"scenario": "MV lookup by customer_id", "execution_ms": lookup_ms},
    {"scenario": "MV top-10, no monetary index", "execution_ms": topn_ms_a},
    {"scenario": "MV top-10, index (monetary DESC)", "execution_ms": topn_ms_b},
    {"scenario": "MV top-10, index (monetary DESC NULLS LAST)", "execution_ms": topn_ms_c},
])
summary["speedup_vs_full"] = (full_ms / summary.execution_ms).round(0)
print("Сортування/спіли у full pipeline:")
for line in plan_lines(full_plan, "Sort Method", "temp read", "Hash Aggregate", "HashAggregate", "WindowAgg", "CTE Scan")[:12]:
    print("  ", line)
summary

Сортування/спіли у full pipeline:
   Buffers: shared hit=15351, temp read=10662 written=10820
   Buffers: shared hit=5609, temp read=1546 written=1550
   Buffers: shared hit=5609, temp read=1546 written=1550
   Buffers: shared hit=5609, temp read=1546 written=1550
   Sort Method: external merge  Disk: 6208kB
   Buffers: shared hit=5609, temp read=1546 written=1550
   Worker 0:  Sort Method: external merge  Disk: 6160kB
   Sort Method: top-N heapsort  Memory: 29kB
   Buffers: shared hit=12587, temp read=9892 written=10048
   Buffers: shared hit=12587, temp read=9892 written=10048
   Buffers: shared hit=11137, temp read=8643 written=8799
   Buffers: shared hit=4654, temp read=7647 written=7801


,scenario,execution_ms,speedup_vs_full
0,full pipeline (9 stages),3708.790,1.0
1,MV lookup by customer_id,0.048,77266.0
2,"MV top-10, no monetary index",31.498,118.0
3,"MV top-10, index (monetary DESC)",30.168,123.0
4,"MV top-10, index (monetary DESC NULLS LAST)",0.055,67433.0


**Висновок за планами.**
- **Full pipeline: ~2 с** (1,9–2,3 с у різних прогонах).
  - Stage 1 матеріалізовано один раз (`CTE stage1_orders_total`, ~0,55 с): Parallel Seq Scan трьох таблиць → Parallel Hash Join → `Sort` (external merge, ~6 MB на диск) → GroupAggregate. Його читають три `CTE Scan` (RFM, rolling AOV, cancel).
  - Далі RFM (`HashAggregate`), cancel (`HashAggregate`) і preferred category з'єднуються ланцюжком `Merge Left Join`. Кожен Merge Join вимагає свого `Sort` за `customer_unique_id`, і майже всі вони йдуть у `external merge`. Разом ~10,8 тис. temp-блоків (~85 MB).
- **MV lookup: < 0,1 мс** — `Index Scan using idx_mv_olist_customer_features_customer_id`, 8 буферів загалом. Це на 4 порядки швидше за full pipeline.
- **Top-10 з MV:**
  - без індексу — Parallel Seq Scan + `top-N heapsort` (~20–25 мс);
  - з індексом **із шаблону** `(monetary DESC)` — **той самий план** з Seq Scan, бо індекс `DESC NULLS FIRST` не відповідає `ORDER BY ... DESC NULLS LAST`;
  - лише з `(monetary DESC NULLS LAST)` — `Index Scan` за ~0,05 мс.
- **Найдорожча ознака — `preferred_category`.** Її піддерево в повному плані займає ~0,6 с, стільки ж, скільки вся спільна база Stage 1. Причина: Stage 6 **не перевикористовує** Stage 1, а заново з'єднує orders × customers × order_items × products (ще одна серія Parallel Hash Join), сортує 112 тис. позицій (`external merge`) для GroupAggregate і ще раз сортує для `WindowAgg` (`ROW_NUMBER`). Другим за вартістю є rolling AOV: два `WindowAgg` над повторно відсортованим Stage 1. Ізольовані виміри в 5.4 підтверджують порядок величин: `preferred_category` окремо коштує стільки ж, скільки вся база Stage 1. Але вони **не адитивні**: `cohort_quarter`, який логічно додає лише `DATE_TRUNC`, стабільно виходить дорожчим за RFM, бо ізольований запит `SELECT r.* ...` planner планує інакше. Тому висновок спирається на вузли всередині одного плану.

### 5.5. Внесок спілу на диск: той самий pipeline з `work_mem = 64MB`

У плані 5.1 майже всі `Sort` мають `Sort Method: external merge  Disk: …`. За `work_mem = 4MB` (значення за замовчуванням) сортування ~100 тис. рядків не вміщується в пам'ять і пишеться в тимчасові файли (`temp read/written` у `Buffers`). `SET LOCAL` змінює параметр лише для однієї транзакції з `EXPLAIN`.

In [29]:
def spill_stats(plan: str) -> dict:
    # Count disk-spilling sorts and temp blocks written at the top node of a plan
    temp = re.search(r"temp read=(\d+) written=(\d+)", plan)
    return {
        "external_merge_sorts": plan.count("external merge"),
        "in_memory_quicksorts": len(re.findall(r"Sort Method: (?:quicksort|top-N heapsort)", plan)),
        "temp_blocks_written": int(temp.group(2)) if temp else 0,
    }


full_runs_4mb = [explain(FULL_SQL, show=False)[1] for _ in range(3)]
plan_64mb, _ = explain(FULL_SQL, show=False, settings="SET LOCAL work_mem = '64MB'")
full_runs_64mb = [explain(FULL_SQL, show=False, settings="SET LOCAL work_mem = '64MB'")[1] for _ in range(3)]

pd.DataFrame([
    {"work_mem": "4MB (default)", "median_ms": round(statistics.median(full_runs_4mb), 1), **spill_stats(full_plan)},
    {"work_mem": "64MB", "median_ms": round(statistics.median(full_runs_64mb), 1), **spill_stats(plan_64mb)},
])

,work_mem,median_ms,external_merge_sorts,in_memory_quicksorts,temp_blocks_written
0,4MB (default),4055.8,7,3,10820
1,64MB,3307.9,0,6,0


**Висновок.** З `work_mem = 64MB` зникають **усі 7** спілів на диск (`external merge` → quicksort у пам'яті, temp-блоків 10 821 → 0). Проте час падає лише на кілька відсотків (2–8 % у різних прогонах). Тимчасові файли такого розміру лежать у кеші ОС, тож головна вартість — у самих JOIN, агрегаціях і сортуваннях на ~100 тис. рядків, а не в дисковому I/O. Тюнінг параметрів не замінює кешування: навіть «ідеальний» перерахунок на 4 порядки повільніший за lookup з MV.

### 5.6. Reflection

**Найдорожча ознака** — `preferred_category`. У плані її піддерево займає ~0,6 с з ~2 с: Stage 6 заново з'єднує чотири таблиці через Parallel Hash Join, сортує 112 тис. позицій (`external merge`) для GroupAggregate і ще раз для `WindowAgg`. Наступний за вартістю — rolling AOV (два `WindowAgg` над пересортованим Stage 1). Збільшення `work_mem` прибрало всі спіли, але дало лише кілька відсотків, тож вартість визначають самі операції, а не I/O.

**Кешувати через MV — так.** Lookup за `customer_id` — це `Index Scan` за < 0,1 мс проти біля 2 с перерахунку. Ознаки мають денну грануляність (recency в днях, когорта — квартал), тому refresh **щодоби** після нічного завантаження замовлень. Щогодини — лише якщо `cancellation_rate` використовується для антифроду. `CONCURRENTLY` повільніший (~3 проти ~2 с), але не блокує читачів, а для serving це важливіше.

**Звичайний VIEW** був би правильнішим там, де потрібна свіжість у момент запиту і рідкісні виклики: ad-hoc аналітика, перевірка одного клієнта в саппорті, дебаг логіки stage-ів.

**Termination recursive CTE:** рекурсія зупиняється, коли recursive part повертає 0 рядків. При циклі A → B → A path-guard `NOT next = ANY(path_ids)` відкидає повторний вузол, depth guard обмежує глибину, а `CYCLE`-clause явно позначає цикл. Без path-guard запит повернув 8 рядків замість 3. А з anchor «без вхідних ребер» цикл узагалі невидимий, тож потрібна перевірка покриття.

**Головний trade-off — latency vs freshness.** Storage дешевий (22 MB), recompute-cost прийнятний для нічного batch. Але `reference_date` з `MAX(order_ts)` змушує щодня перераховувати recency для всіх рядків, тож свіжість коштує повного refresh. Для online-моделей я зберігала б у MV `last_order_date`, а `recency_days` рахувала б під час запиту.